## 1. Load EPA NO2 Bronze Data

Load raw EPA NO2 measurements from four JSON files covering Bronx and Queens for June and July 2025.

Inspect the record count and schema before applying Silver-layer transformations.

In [ ]:
bronze_path = "Files/bronze/epa_no2/"

files = [
    "epa_no2_bronx_202506.json",
    "epa_no2_bronx_202507.json",
    "epa_no2_queens_202506.json",
    "epa_no2_queens_202507.json"
]

file_paths = [bronze_path + f for f in files]

df_bronze = (
    spark.read
    .option("multiLine", "true")
    .json(file_paths)
)

print("Total rows:", df_bronze.count())
df_bronze.printSchema()

## 2. Select Required EPA Columns

Select the station identifiers, geographic coordinates, timestamps, NO2 measurements, units, and quality qualifiers required for Silver-layer processing.

In [ ]:
df_selected = df_bronze.select(
    "state_code",
    "county_code",
    "site_number",
    "poc",
    "latitude",
    "longitude",
    "date_local",
    "time_local",
    "date_gmt",
    "time_gmt",
    "sample_measurement",
    "units_of_measure",
    "qualifier"
)

df_selected.show(5, truncate=False)

## 3. Validate and Filter EPA NO2 Measurements

Analyze EPA quality qualifiers and missing measurement values.

Retain only records with valid NO2 measurements and no quality qualifiers before further Silver-layer processing.

In [ ]:
from pyspark.sql import functions as F

df_selected.groupBy("qualifier").agg(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(
            F.col("sample_measurement").isNull(), 1
        ).otherwise(0)
    ).alias("null_measurements")
).orderBy(
    F.desc("total_rows")
).show(20, truncate=False)

In [ ]:
df_valid = df_selected.filter(
    F.col("sample_measurement").isNotNull()
    & F.col("qualifier").isNull()
)

print("Valid rows:", df_valid.count())
print("Removed rows:", df_selected.count() - df_valid.count())

## 4. Validate Duplicate EPA Measurements

Check for duplicate NO2 measurements using station identifiers, measurement channel (POC), and UTC date/time to ensure data uniqueness before Silver-layer transformation.

In [ ]:
df_duplicates = (
    df_valid
    .groupBy(
        "state_code",
        "county_code",
        "site_number",
        "poc",
        "date_gmt",
        "time_gmt"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate groups:", df_duplicates.count())

df_duplicates.show(10, truncate=False)

## 5. Convert and Validate EPA Timestamps

Convert EPA date and time fields into Spark timestamp and date types.

Use UTC timestamps for consistent time-based joins and local dates for daily reporting. Validate the converted fields for NULL values.

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

df_typed = (
    df_valid
    .withColumn(
        "measurement_utc",
        F.to_timestamp(
            F.concat_ws(" ", "date_gmt", "time_gmt"),
            "yyyy-MM-dd HH:mm"
        )
    )
    .withColumn(
        "measurement_date_lst",
        F.to_date("date_local", "yyyy-MM-dd")
    )
)

df_typed.select(
    "site_number",
    "date_gmt",
    "time_gmt",
    "measurement_utc",
    "measurement_date_lst",
    "sample_measurement"
).show(5, truncate=False)

In [ ]:
df_typed.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.col("measurement_utc").isNull().cast("int")
    ).alias("null_utc"),
    F.sum(
        F.col("measurement_date_lst").isNull().cast("int")
    ).alias("null_dates"),
    F.sum(
        F.col("sample_measurement").isNull().cast("int")
    ).alias("null_no2")
).show()

## 6. Generate Station ID

Create a unique station identifier using state, county, and site codes.

Validate the number of NO2 measurements for each monitoring station.

In [ ]:
df_silver = df_typed.withColumn(
    "station_id",
    F.concat_ws(
        "-",
        F.col("state_code"),
        F.col("county_code"),
        F.col("site_number")
    )
)

df_silver.groupBy(
    "station_id",
    "county_code"
).count().orderBy(
    "county_code",
    "station_id"
).show(truncate=False)

## 7. Validate NO2 Measurement Units and Values

Verify that NO2 measurements use consistent units and check the minimum, maximum, and negative values before saving the Silver table.

In [ ]:
df_silver.groupBy("units_of_measure").agg(
    F.count("*").alias("total_rows"),
    F.min("sample_measurement").alias("min_no2"),
    F.max("sample_measurement").alias("max_no2"),
    F.sum(
        F.when(F.col("sample_measurement") < 0, 1)
         .otherwise(0)
    ).alias("negative_values")
).show(truncate=False)

## 8. Prepare and Save Silver EPA NO2

Select the cleaned EPA NO2 measurements and standardize the concentration column as no2_ppb.

Save the final dataset as a Silver Delta table and verify the persisted record count.

In [ ]:
df_silver_final = df_silver.select(
    "station_id",
    "state_code",
    "county_code",
    "site_number",
    "poc",
    "latitude",
    "longitude",
    "measurement_utc",
    "measurement_date_lst",
    "time_local",
    F.col("sample_measurement").alias("no2_ppb")
)

In [ ]:
(
    df_silver_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_epa_no2_hourly")
)

print(
    "Saved rows:",
    spark.table("silver_epa_no2_hourly").count()
)